# SRoBERTa-NLI-base — Training + Table 1 STS Evaluation (combined, Kaggle-ready)

This notebook combines `01_SBERT_NLI_Training.ipynb` and `02_SBERT_STS_Evaluation.ipynb`
into a single pipeline, swapped to **`roberta-base`** as the backbone, to reproduce the
paper's **SRoBERTa-NLI-base** row of Table 1.

```text
Pretrained RoBERTa-base
      ↓
Mean Pooling
      ↓
Siamese / tied-weight encoder
      ↓
[u ; v ; |u-v|]
      ↓
3-class NLI classifier   (Sec. 3.1: trained for 1 epoch, batch 16, Adam 2e-5, 10% warm-up)
      ↓
SNLI + MultiNLI
      ↓
discard classifier, keep RoBERTa + mean pooling
      ↓
cosine similarity on STS12-16, STS-B, SICK-R
      ↓
Spearman correlation  ->  Table 1
```

**Target (paper Table 1, `SRoBERTa-NLI-base`):**

| Dataset | Paper |
|---|---:|
| STS12 | 71.54 |
| STS13 | 72.49 |
| STS14 | 70.80 |
| STS15 | 78.74 |
| STS16 | 73.69 |
| STS-B | 77.77 |
| SICK-R | 74.46 |
| **Average** | **74.21** |

### What's different from the BERT-base version
1. **Backbone is `roberta-base`, not `bert-base-uncased`.** Everything else (mean pooling,
   `[u; v; |u-v|]` classifier head, batch/LR/warmup schedule) is unchanged - the paper explicitly
   states SRoBERTa uses the same siamese fine-tuning recipe as SBERT (Sec. 3.1/3).
2. **No `token_type_ids`.** RoBERTa's tokenizer does not produce segment IDs (it has no
   next-sentence-prediction pretraining objective, so it has no token-type embedding table).
   The `encode()` method already guards with `.get("token_type_ids")`, so this works unchanged -
   RoBERTa's tokenizer simply never returns that key and it's never passed to the model.
3. **Training and evaluation are one notebook**, run top to bottom. The trained encoder is kept
   in memory and used directly for the STS evaluation section - no save/reload round-trip
   required, which removes a class of "wrong prefix / wrong checkpoint" bugs. A checkpoint is
   still written to `outputs/` for the record (and so Kaggle keeps it as a notebook output when
   you commit).
4. **Data path resolution is Kaggle-aware.** The original notebooks assumed `Path("..")/"data"`
   (notebook/ and data/ as sibling folders on a local machine). On Kaggle, your uploaded dataset
   is mounted read-only under `/kaggle/input/<dataset-name>/...` and has nothing to do with the
   notebook's working directory - `Path("..")/"data"` will not resolve there. The `resolve_data_root()`
   helper below searches, in order: `../data`, `./data`, then every folder under `/kaggle/input/`
   for one that contains `NLI` and `STS` subfolders, so it works whether your `data/` and
   `notebook/` folders are siblings locally or `data/` sits inside a Kaggle Dataset input.


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm
from scipy.stats import spearmanr

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

In [ ]:
# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False

print("Seed:", SEED)

In [ ]:
# ============================================================
# 3. KAGGLE-AWARE DATA PATH RESOLUTION
# ============================================================

KNOWN_KAGGLE_DATA_PATHS = [
    Path("/kaggle/input/datasets/adityammantri/data-table2/data"),
]


def looks_like_data_root(p):
    return (p / "NLI").exists() and (p / "STS").exists()


def search_kaggle_input(max_depth=4):
    root = Path("/kaggle/input")
    if not root.exists():
        return None

    frontier = [(root, 0)]
    while frontier:
        current, depth = frontier.pop(0)
        try:
            children = [c for c in current.iterdir() if c.is_dir()]
        except PermissionError:
            continue

        if looks_like_data_root(current):
            return current

        if depth < max_depth:
            frontier.extend((c, depth + 1) for c in children)

    return None


def resolve_data_root():
    candidates = [Path("../data"), Path("data")] + KNOWN_KAGGLE_DATA_PATHS

    for c in candidates:
        if looks_like_data_root(c):
            return c

    discovered = search_kaggle_input()
    if discovered is not None:
        return discovered

    raise FileNotFoundError(
        "Could not locate a data directory containing both 'NLI' and 'STS' subfolders.\n"
        "Checked known candidates:\n"
        + "\n".join(f"  - {c.resolve() if c.exists() else c}" for c in candidates)
        + "\n\nAlso searched recursively under /kaggle/input and found nothing.\n\n"
        "On Kaggle: attach your dataset as a Notebook input, and make sure it contains\n"
        "a folder with 'NLI' and 'STS' subfolders somewhere inside it."
    )


DATA_DIR = resolve_data_root()
print("Resolved data root:", DATA_DIR.resolve())

NLI_PATH = DATA_DIR / "NLI" / "NLI"
STS_DIR = DATA_DIR / "STS"

if not NLI_PATH.exists():
    alt = DATA_DIR / "NLI"
    if (alt / "dataset_info.json").exists() or (alt / "dataset_dict.json").exists() or any(alt.glob("*.arrow")):
        NLI_PATH = alt

print("NLI path:", NLI_PATH.resolve())
print("STS dir:", STS_DIR.resolve())

In [ ]:
# ============================================================
# 4. CONFIGURATION
# ============================================================

MODEL_NAME = "roberta-base"

OUTPUT_DIR = Path("outputs") / "sroberta_nli_base"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Paper configuration (Sec. 3.1) - identical for SBERT and SRoBERTa.
EPOCHS = 1
LEARNING_RATE = 2e-5
PAPER_BATCH_SIZE = 16
WARMUP_RATIO = 0.10

# Hardware-adapted implementation (not a paper claim):
# 4 physical examples x 4 accumulation steps = effective batch size 16.
PHYSICAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = PAPER_BATCH_SIZE // PHYSICAL_BATCH_SIZE

# The paper text does not specify max sequence length.
MAX_LENGTH = 128

NUM_LABELS = 3
EVAL_BATCH_SIZE = 32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()

print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Physical batch size:", PHYSICAL_BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", PHYSICAL_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)
print("Warmup ratio:", WARMUP_RATIO)
print("Max length:", MAX_LENGTH)

# Part 1 — NLI Training

In [ ]:
# ============================================================
# 5. LOAD THE COMBINED NLI DATASET
# ============================================================

if not NLI_PATH.exists():
    raise FileNotFoundError(
        f"Combined NLI dataset not found at: {NLI_PATH.resolve()}\n"
        "Run data.py first, or check the resolved DATA_DIR above."
    )

nli_dataset = load_from_disk(str(NLI_PATH))

print(nli_dataset)
print()
print("Number of examples:", f"{len(nli_dataset):,}")
print("Columns:", nli_dataset.column_names)

In [ ]:
# ============================================================
# 6. FINAL NLI SANITY CHECK
# ============================================================

labels = np.asarray(nli_dataset["label"])

print("Unique labels:", sorted(np.unique(labels).tolist()))
print("Label counts:")
print(pd.Series(labels).value_counts().sort_index())

assert set(np.unique(labels)).issubset({0, 1, 2}), "Invalid NLI labels found."
assert len(nli_dataset) > 0, "NLI dataset is empty."

print()
print("NLI dataset is valid for 3-class training.")

In [ ]:
# ============================================================
# 7. TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer:", MODEL_NAME)
print("Vocabulary size:", tokenizer.vocab_size)
print("PAD token:", tokenizer.pad_token)
print("CLS/BOS token:", tokenizer.cls_token)
print("SEP/EOS token:", tokenizer.sep_token)
print("Model input names:", tokenizer.model_input_names)
print(
    "-> Note: RoBERTa's tokenizer does not produce 'token_type_ids' "
    "(no next-sentence-prediction segment embeddings in RoBERTa's pretraining). "
    "This is expected and handled by encode() below via .get('token_type_ids')."
)

In [ ]:
# ============================================================
# 8. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):
    """Tokenize a batch of NLI sentence pairs with dynamic padding."""

    premises = [example["premise"] for example in batch]
    hypotheses = [example["hypothesis"] for example in batch]
    labels = torch.tensor(
        [example["label"] for example in batch],
        dtype=torch.long
    )

    premise_inputs = tokenizer(
        premises,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    hypothesis_inputs = tokenizer(
        hypotheses,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    return {
        "premise": premise_inputs,
        "hypothesis": hypothesis_inputs,
        "labels": labels,
    }


train_loader = DataLoader(
    nli_dataset,
    batch_size=PHYSICAL_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

print("Number of physical batches:", len(train_loader))

In [ ]:
# ============================================================
# 9. MEAN POOLING
# ============================================================

def mean_pooling(last_hidden_state, attention_mask):
    """Mean-pool token representations while ignoring padding."""

    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

In [ ]:
# ============================================================
# 10. SROBERTA NLI MODEL
# ============================================================

class SRoBERTaForNLI(nn.Module):

    def __init__(self, model_name, num_labels=3):
        super().__init__()

        # One RoBERTa encoder is shared by both sentences (siamese / tied weights).
        self.encoder = AutoModel.from_pretrained(model_name)

        self.hidden_size = self.encoder.config.hidden_size

        self.classifier = nn.Linear(
            self.hidden_size * 3,
            num_labels
        )

    def encode(self, input_ids, attention_mask, token_type_ids=None):

        kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }

        # RoBERTa's tokenizer never returns token_type_ids, so this is always
        # None for RoBERTa and simply skipped - kept for interface parity
        # with the BERT version of this model.
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        outputs = self.encoder(**kwargs)

        embedding = mean_pooling(
            outputs.last_hidden_state,
            attention_mask
        )

        return embedding

    def forward(self, premise, hypothesis):

        u = self.encode(
            input_ids=premise["input_ids"],
            attention_mask=premise["attention_mask"],
            token_type_ids=premise.get("token_type_ids"),
        )

        v = self.encode(
            input_ids=hypothesis["input_ids"],
            attention_mask=hypothesis["attention_mask"],
            token_type_ids=hypothesis.get("token_type_ids"),
        )

        difference = torch.abs(u - v)

        combined = torch.cat([u, v, difference], dim=1)

        logits = self.classifier(combined)

        return logits, u, v


model = SRoBERTaForNLI(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS
)

model.to(DEVICE)

print(model.classifier)
print()
print("RoBERTa hidden size:", model.hidden_size)
print("Classifier input size:", model.classifier.in_features)

In [ ]:
# ============================================================
# 11. VERIFY ARCHITECTURE
# ============================================================

encoder_parameter_count = sum(p.numel() for p in model.encoder.parameters())
total_parameter_count = sum(p.numel() for p in model.parameters())

print(f"RoBERTa parameters: {encoder_parameter_count:,}")
print(f"Total parameters:   {total_parameter_count:,}")

assert model.classifier.in_features == 2304

print()
print("Architecture sanity check passed.")

In [ ]:
# ============================================================
# 12. TEST ONE FORWARD PASS
# ============================================================

batch = next(iter(train_loader))

batch_premise = {k: v.to(DEVICE) for k, v in batch["premise"].items()}
batch_hypothesis = {k: v.to(DEVICE) for k, v in batch["hypothesis"].items()}
batch_labels = batch["labels"].to(DEVICE)

model.eval()

with torch.no_grad():
    logits, u, v = model(batch_premise, batch_hypothesis)

print("Premise embedding shape:", tuple(u.shape))
print("Hypothesis embedding shape:", tuple(v.shape))
print("Logits shape:", tuple(logits.shape))
print("Labels shape:", tuple(batch_labels.shape))

assert u.shape[1] == model.hidden_size
assert v.shape[1] == model.hidden_size
assert logits.shape[1] == 3

print()
print("Forward pass passed.")

In [ ]:
# ============================================================
# 13. LOSS, OPTIMIZER, AND SCHEDULER
# ============================================================

criterion = nn.CrossEntropyLoss()

# The paper specifies Adam with learning rate 2e-5.
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

steps_per_epoch = math.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS)
total_optimizer_steps = steps_per_epoch * EPOCHS
warmup_steps = int(total_optimizer_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps,
)

print("Physical batches:", len(train_loader))
print("Optimizer steps:", total_optimizer_steps)
print("Warmup steps:", warmup_steps)

In [ ]:
# ============================================================
# 14. TRAINING LOOP
# ============================================================

def train_one_epoch(
    model,
    dataloader,
    optimizer,
    scheduler,
    criterion,
    device,
    accumulation_steps,
    use_amp=True,
):
    model.train()

    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    progress = tqdm(dataloader, desc="Training", leave=True)

    for step, batch in enumerate(progress):

        premise = {k: v.to(device, non_blocking=True) for k, v in batch["premise"].items()}
        hypothesis = {k: v.to(device, non_blocking=True) for k, v in batch["hypothesis"].items()}
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits, _, _ = model(premise, hypothesis)

        # Loss computed in fp32, outside autocast - cross-entropy is fairly
        # fp16-safe, but keeping it consistent/robust with the rest of this
        # project's notebooks costs nothing here.
        loss = criterion(logits.float(), labels)
        loss_for_backward = loss / accumulation_steps

        scaler.scale(loss_for_backward).backward()

        running_loss += loss.item()

        is_last_batch = (step + 1) == len(dataloader)

        if (step + 1) % accumulation_steps == 0 or is_last_batch:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        if (step + 1) % 100 == 0:
            average_loss = running_loss / (step + 1)
            progress.set_postfix(loss=f"{average_loss:.4f}")

    return running_loss / len(dataloader)

In [ ]:
# ============================================================
# 15. TRAIN SROBERTA FOR ONE EPOCH
# ============================================================

training_history = []

for epoch in range(EPOCHS):
    print()
    print("=" * 70)
    print(f"EPOCH {epoch + 1}/{EPOCHS}")
    print("=" * 70)

    avg_loss = train_one_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=DEVICE,
        accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        use_amp=USE_AMP,
    )

    training_history.append({
        "epoch": epoch + 1,
        "loss": avg_loss,
        "learning_rate": scheduler.get_last_lr()[0],
    })

    print(f"Average training loss: {avg_loss:.4f}")

history_df = pd.DataFrame(training_history)
display(history_df)
print()
print("Training completed.")

In [ ]:
# ============================================================
# 16. SAVE THE SROBERTA MODEL (for the record / Kaggle output)
# ============================================================

MODEL_OUTPUT = OUTPUT_DIR / "model"
TOKENIZER_OUTPUT = OUTPUT_DIR / "tokenizer"

MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)
TOKENIZER_OUTPUT.mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "model_name": MODEL_NAME,
        "pooling": "mean",
        "num_labels": NUM_LABELS,
        "max_length": MAX_LENGTH,
        "seed": SEED,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE,
        "effective_batch_size": PAPER_BATCH_SIZE,
        "physical_batch_size": PHYSICAL_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    },
    MODEL_OUTPUT / "sroberta_nli_base.pt"
)

tokenizer.save_pretrained(str(TOKENIZER_OUTPUT))

history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

print("Saved model to:", MODEL_OUTPUT.resolve())
print("Saved tokenizer to:", TOKENIZER_OUTPUT.resolve())

# Part 2 — Table 1 STS Evaluation

The NLI classifier head is discarded. We reuse `model.encoder` (RoBERTa + mean pooling)
**directly from memory** - no save/reload round-trip needed since training and evaluation
are in the same notebook/session.

In [ ]:
# ============================================================
# 17. BUILD THE STS SENTENCE ENCODER FROM THE TRAINED MODEL
# ============================================================

encoder = model.encoder  # trained RoBERTa weights, still in memory
encoder.eval()

print("Encoder ready. Hidden size:", encoder.config.hidden_size)
print("Device:", next(encoder.parameters()).device)

In [ ]:
# ============================================================
# 18. SENTENCE ENCODER FUNCTION
# ============================================================

@torch.no_grad()
def encode_sentences(sentences, batch_size=EVAL_BATCH_SIZE):
    all_embeddings = []

    for start in tqdm(range(0, len(sentences), batch_size), desc="Encoding"):
        batch = sentences[start:start + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )

        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

        outputs = encoder(**inputs)

        embeddings = mean_pooling(outputs.last_hidden_state, inputs["attention_mask"])

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings, dim=0)

In [ ]:
# ============================================================
# 19. ENCODER SANITY CHECK
# ============================================================

test_sentences = [
    "A man is eating food.",
    "A person is having a meal."
]

test_embeddings = encode_sentences(test_sentences)

print("Embedding shape:", test_embeddings.shape)
print("Expected: (2,", encoder.config.hidden_size, ")")

u = test_embeddings[0].numpy()
v = test_embeddings[1].numpy()

cosine = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12)
print("Cosine similarity:", cosine)

In [ ]:
# ============================================================
# 20. STS EVALUATION FUNCTION
# ============================================================

def evaluate_sts_dataframe(df, name):
    sentences1 = df["sentence1"].astype(str).tolist()
    sentences2 = df["sentence2"].astype(str).tolist()
    gold = df["score"].astype(float).to_numpy()

    print(f"\n{name}")
    print("=" * len(name))
    print("Pairs:", len(df))

    emb1 = encode_sentences(sentences1).numpy()
    emb2 = encode_sentences(sentences2).numpy()

    numerator = np.sum(emb1 * emb2, axis=1)
    denominator = np.linalg.norm(emb1, axis=1) * np.linalg.norm(emb2, axis=1)

    predicted = numerator / (denominator + 1e-12)

    rho = spearmanr(predicted, gold).statistic
    score = rho * 100

    print(f"Spearman rho: {rho:.6f}")
    print(f"Paper-style score: {score:.2f}")

    return {"dataset": name, "score": score, "spearman": rho, "n_pairs": len(df)}

In [ ]:
# ============================================================
# 21. LOAD ALL STS DATASETS
# ============================================================

STS_DATASETS = {
    "STS12": STS_DIR / "STS12",
    "STS13": STS_DIR / "STS13",
    "STS14": STS_DIR / "STS14",
    "STS15": STS_DIR / "STS15",
    "STS16": STS_DIR / "STS16",
    "STS-B": STS_DIR / "STSb",
    "SICK-R": STS_DIR / "SICK-R",
}

sts_datasets = {}

for name, path in STS_DATASETS.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print("Path:", path.resolve())

    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path.resolve()}")

    dataset = load_from_disk(path)
    sts_datasets[name] = dataset

    print(dataset)

In [ ]:
# ============================================================
# 22. NORMALIZE STS DATASETS
# ============================================================

def get_test_split(dataset):
    """Preference: test > validation > train."""
    if hasattr(dataset, "keys"):
        if "test" in dataset:
            return dataset["test"]
        if "validation" in dataset:
            return dataset["validation"]
        if "train" in dataset:
            return dataset["train"]
    return dataset


def normalize_sts_dataset(dataset, name):
    dataset = get_test_split(dataset)

    print(f"\n{name}")
    print("-" * len(name))
    print("Columns:", dataset.column_names)
    print("Examples:", len(dataset))

    columns = dataset.column_names

    sentence1_candidates = ["sentence1", "sentence_1", "sent1", "text1"]
    sentence2_candidates = ["sentence2", "sentence_2", "sent2", "text2"]
    score_candidates = ["score", "similarity", "relatedness_score", "label"]

    sentence1_col = next((c for c in sentence1_candidates if c in columns), None)
    sentence2_col = next((c for c in sentence2_candidates if c in columns), None)
    score_col = next((c for c in score_candidates if c in columns), None)

    if sentence1_col is None:
        raise ValueError(f"{name}: Could not find sentence1 column.\nAvailable columns: {columns}")
    if sentence2_col is None:
        raise ValueError(f"{name}: Could not find sentence2 column.\nAvailable columns: {columns}")
    if score_col is None:
        raise ValueError(f"{name}: Could not find score column.\nAvailable columns: {columns}")

    df = dataset.to_pandas()
    df = df[[sentence1_col, sentence2_col, score_col]].copy()
    df.columns = ["sentence1", "sentence2", "score"]

    return df


sts_data = {}
for name, dataset in sts_datasets.items():
    sts_data[name] = normalize_sts_dataset(dataset, name)

In [ ]:
# ============================================================
# 23. VERIFY ALL STS DATASETS
# ============================================================

for name, df in sts_data.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print("Shape:", df.shape)
    print("Missing values:")
    print(df.isna().sum())

In [ ]:
# ============================================================
# 24. EVALUATE ALL SEVEN STS DATASETS
# ============================================================

results = []

for name in ["STS12", "STS13", "STS14", "STS15", "STS16", "STS-B", "SICK-R"]:
    result = evaluate_sts_dataframe(sts_data[name], name)
    results.append(result)

In [ ]:
# ============================================================
# 25. TABLE 1 COMPARISON — SRoBERTa-NLI-base
# ============================================================

paper_results = {
    "STS12": 71.54,
    "STS13": 72.49,
    "STS14": 70.80,
    "STS15": 78.74,
    "STS16": 73.69,
    "STS-B": 77.77,
    "SICK-R": 74.46,
}

our_df = pd.DataFrame(results)

paper_df = pd.DataFrame(
    list(paper_results.items()),
    columns=["dataset", "paper_score"]
)

comparison = paper_df.merge(
    our_df[["dataset", "score"]],
    on="dataset",
    how="left"
)

comparison["difference"] = comparison["score"] - comparison["paper_score"]

display(comparison)

comparison.to_csv(OUTPUT_DIR / "table1_sroberta_nli_base_comparison.csv", index=False)

In [ ]:
# ============================================================
# 26. AVERAGE
# ============================================================

if comparison["score"].notna().all():
    our_average = comparison["score"].mean()
    paper_average = comparison["paper_score"].mean()

    print("Our average:", f"{our_average:.2f}")
    print("Paper average (74.21):", f"{paper_average:.2f}")
    print("Average difference:", f"{our_average - paper_average:.2f}")
else:
    print("Some datasets were not evaluated.")